Import Necessary Libraries

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import re
import numpy as np
from gensim.models import Word2Vec
from tqdm import tqdm
import requests

GITHUB Details - Replace GITHUB ACCESS KEY with your own Personal access token

In [ ]:
github_token = "GITHUB ACCESS KEY"
headers = {"Authorization": f"token {github_token}"}

Function for Text pre-processing

In [ ]:
def clean_text(text):
    text = re.sub(r'`.*?`', '', text)
    text = re.sub(r'[\\U00010000-\\U0010ffff]', '', text)
    text = re.sub(r'[\\W]+', ' ', text)
    return text.lower().strip()

# Tokenize text for Word2Vec
def tokenize_text(text):
    return text.split()

# Convert text to vector by averaging word vectors
def text_to_vector(model, tokens, vector_size):
    vec = np.zeros(vector_size)
    count = 0
    for word in tokens:
        if word in model.wv:
            vec += model.wv[word]
            count += 1
    return vec / count if count > 0 else vec

Discussion to Issues

In [ ]:
df = pd.read_csv("../Dataset/DiscussionToIssue.csv")

In [ ]:
encoder = LabelEncoder()
df['IsIssueRaised'] = encoder.fit_transform(df['IsIssueRaised'])
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

Training ...

In [ ]:
df_shuffled['concatenated'] = (df_shuffled['Title'] + ' ' + df_shuffled['Description'] + ' ' + df_shuffled['Comments']).apply(clean_text)

df_shuffled['tokens'] = df_shuffled['concatenated'].apply(tokenize_text)
w2v_model = Word2Vec(sentences=df_shuffled['tokens'], vector_size=100, window=5, min_count=1, workers=4, seed=42)

# Convert all texts to vectors
X = np.array([text_to_vector(w2v_model, tokens, 100) for tokens in df_shuffled['tokens']])
y = df_shuffled['IsIssueRaised'].values


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

Testing ...

In [ ]:
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Discussion to Issues with Description Alone

In [ ]:
df_shuffled['concatenated'] = (df_shuffled['Title'] + ' ' + df_shuffled['Description']).apply(clean_text)

df_shuffled['tokens'] = df_shuffled['concatenated'].apply(tokenize_text)
w2v_model = Word2Vec(sentences=df_shuffled['tokens'], vector_size=100, window=5, min_count=1, workers=4, seed=42)

# Convert all texts to vectors
X = np.array([text_to_vector(w2v_model, tokens, 100) for tokens in df_shuffled['tokens']])
y = df_shuffled['IsIssueRaised'].values

Training ...

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

Testing ...

In [ ]:
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Discussion to Issues with First Comment alone

Title + Description + First Comment

Function to extract Repository and Discussion number

In [ ]:
def extract_github_path(url):
    match = re.search(r'github\.com/([^?#]*)', url)
    return match.group(1) if match else None

Fetch the First comment from Discussion 

In [ ]:
df_shuffled['Comment'] =  None
for index,row in tqdm(df_shuffled.iterrows()):
  repo = extract_github_path(row['Issue'])
  curl = f'https://api.github.com/repos/{repo}/comments'
  repo_comment=[]
  cresponse = requests.get(curl,  headers=headers)
  if cresponse.status_code == 200:
    issueComments = cresponse.json()
    cnt=0
    for comment in issueComments:
        repo_comment.append(comment['body'])
        cnt+=1
        if(cnt<1):
          break
  df_shuffled.at[index, 'Comment'] = repo_comment

Training ...

In [ ]:
df_shuffled['concatenated'] = (df_shuffled['Title'] + ' ' + df_shuffled['Description']+' '+str(df_shuffled['Comment'])).apply(clean_text)
df_shuffled['tokens'] = df_shuffled['concatenated'].apply(tokenize_text)
w2v_model = Word2Vec(sentences=df_shuffled['tokens'], vector_size=100, window=5, min_count=1, workers=4, seed=42)

# Convert all texts to vectors
X = np.array([text_to_vector(w2v_model, tokens, 100) for tokens in df_shuffled['tokens']])
y = df_shuffled['IsIssueRaised'].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

Testing ...

In [ ]:
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Issues to Discussion

In [ ]:
df1 = pd.read_csv("../Dataset/IssueToDiscussion.csv")

In [ ]:
encoder = LabelEncoder()
df1['ConvertedFromIssue'] = encoder.fit_transform(df1['ConvertedFromIssue'])
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

Training ...

In [ ]:
df1_shuffled['concatenated'] = (df1_shuffled['Title'] + ' ' + df1_shuffled['Description'] + ' ' + df1_shuffled['Comments']).apply(clean_text)

df1_shuffled['tokens'] = df1_shuffled['concatenated'].apply(tokenize_text)
w2v_model = Word2Vec(sentences=df1_shuffled['tokens'], vector_size=100, window=5, min_count=1, workers=4, seed=42)

# Convert all texts to vectors
X = np.array([text_to_vector(w2v_model, tokens, 100) for tokens in df1_shuffled['tokens']])
y = df1_shuffled['ConvertedFromIssue'].values


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

Testing ...

In [ ]:
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Issue to Discussion with Description alone

In [ ]:
df1 = pd.read_csv("../Dataset/IssueToDiscussion.csv")

In [ ]:
encoder = LabelEncoder()
df1['ConvertedFromIssue'] = encoder.fit_transform(df1['ConvertedFromIssue'])
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

Training ...

In [ ]:
df1_shuffled['concatenated'] = (df1_shuffled['Description']).apply(clean_text)

df1_shuffled['tokens'] = df1_shuffled['concatenated'].apply(tokenize_text)
w2v_model = Word2Vec(sentences=df1_shuffled['tokens'], vector_size=100, window=5, min_count=1, workers=4, seed=42)

# Convert all texts to vectors
X = np.array([text_to_vector(w2v_model, tokens, 100) for tokens in df1_shuffled['tokens']])
y = df1_shuffled['ConvertedFromIssue'].values


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

Testing ...

In [ ]:
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Issue to Discussion with Description + First Comment

In [ ]:
df1 = pd.read_csv("../Dataset/IssueToDiscussion.csv")

In [ ]:
encoder = LabelEncoder()
df1['ConvertedFromIssue'] = encoder.fit_transform(df1['ConvertedFromIssue'])
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

Function to extract Repository and Issue number

In [ ]:
def extract_github_path(url):
    match = re.search(r'github\.com/([^?#]*)', url)
    return match.group(1) if match else None

Download the First Comment of an Issue

In [ ]:
df1_shuffled['Comment'] =  None
for index,row in tqdm(df1_shuffled.iterrows()):
  repo = extract_github_path(row['Issue'])
  curl = f'https://api.github.com/repos/{repo}/comments'
  repo_comment=[]
  cresponse = requests.get(curl,  headers=headers)
  if cresponse.status_code == 200:
    issueComments = cresponse.json()
    cnt=0
    for comment in issueComments:
        repo_comment.append(comment['body'])
        cnt+=1
        if(cnt<1):
          break
  df1_shuffled.at[index, 'Comment'] = repo_comment

Training ...

In [ ]:
df1_shuffled['concatenated'] = (df1_shuffled['Description'] + ' ' + str(df1_shuffled['Comment'])).apply(clean_text)

df1_shuffled['tokens'] = df1_shuffled['concatenated'].apply(tokenize_text)
w2v_model = Word2Vec(sentences=df1_shuffled['tokens'], vector_size=100, window=5, min_count=1, workers=4, seed=42)

# Convert all texts to vectors
X = np.array([text_to_vector(w2v_model, tokens, 100) for tokens in df1_shuffled['tokens']])
y = df1_shuffled['ConvertedFromIssue'].values


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

Testing ...

In [ ]:
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Issue to Discussion with Comments

In [ ]:
df1 = pd.read_csv("../Dataset/IssueToDiscussion.csv")

In [ ]:
encoder = LabelEncoder()
df1['ConvertedFromIssue'] = encoder.fit_transform(df1['ConvertedFromIssue'])
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

Training ...

In [ ]:
df1_shuffled['concatenated'] = (df1_shuffled['Comments']).apply(clean_text)

df1_shuffled['tokens'] = df1_shuffled['concatenated'].apply(tokenize_text)
w2v_model = Word2Vec(sentences=df1_shuffled['tokens'], vector_size=100, window=5, min_count=1, workers=4, seed=42)

# Convert all texts to vectors
X = np.array([text_to_vector(w2v_model, tokens, 100) for tokens in df1_shuffled['tokens']])
y = df1_shuffled['ConvertedFromIssue'].values


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

Testing ...

In [ ]:
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))